In [ ]:
# install pymongo and load all csvs
!pip install pymongo -q
from pymongo import MongoClient
import pandas as pd
import time

customers  = pd.read_csv("customers.csv")
orders     = pd.read_csv("orders.csv",     parse_dates=["order_created_at"])
deliveries = pd.read_csv("deliveries.csv", parse_dates=["dispatch_time","delivery_completed_at"])
complaints = pd.read_csv("complaints.csv", parse_dates=["created_at"])
incidents  = pd.read_csv("incidents.csv")
drivers    = pd.read_csv("drivers.csv")
hubs       = pd.read_csv("hubs.csv")
vehicles   = pd.read_csv("vehicles.csv")
app_events = pd.read_csv("app_events.csv")

In [ ]:
# normalise zones before inserting so aggregations produce clean single groups per zone
def norm_zone(s):
    return s.str.strip().str.title().replace("Ctr", "Central")

for df, cols in [
    (orders,     ["pickup_zone", "dropoff_zone"]),
    (customers,  ["home_zone"]),
    (drivers,    ["base_zone"]),
    (hubs,       ["zone"]),
    (app_events, ["zone_context"]),
    (deliveries, ["pickup_zone"])
]:
    for col in cols:
        if col in df.columns:
            df[col] = norm_zone(df[col])

print("zones normalised:", sorted(orders["pickup_zone"].unique()))

In [ ]:
# connect to MongoDB Atlas
MONGO_URI = "mongodb+srv://Henry_Sechler:<mongotron>@henry.vdvy9wz.mongodb.net/?appName=Henry"
client = MongoClient(MONGO_URI)
db = client["northstar"]
print("connected:", db.list_collection_names())

In [ ]:
# wipe and rebuild all collections from scratch
for col in ["customer_profiles", "deliveries", "drivers", "hubs", "app_events"]:
    db[col].drop()
print("cleared:", db.list_collection_names())

In [ ]:
# customer_profiles - embed complaints inside each customer
complaints_by_cust = complaints.groupby("customer_id").apply(
    lambda x: x[["complaint_id","status","complaint_type","severity","created_at"]].to_dict("records"),
    include_groups=False
).to_dict()

customer_docs = []
for _, row in customers.iterrows():
    doc = row.to_dict()
    doc["complaints"] = complaints_by_cust.get(doc["customer_id"], [])
    customer_docs.append(doc)

db.customer_profiles.insert_many(customer_docs)
print(f"customer_profiles: {db.customer_profiles.count_documents({})} docs")

In [ ]:
# deliveries - merge order info in to avoid joins at query time
del_ord = deliveries.merge(orders, on="order_id", how="left")
for col in del_ord.select_dtypes(include=["datetime64[ns]"]).columns:
    del_ord[col] = del_ord[col].replace({pd.NaT: None})

db.deliveries.insert_many(del_ord.to_dict("records"))
print(f"deliveries: {db.deliveries.count_documents({})} docs")

In [ ]:
# drivers - embed each driver's incident history
inc_with_drivers = incidents.merge(deliveries[["delivery_id","driver_id"]], on="delivery_id", how="left")
inc_by_driver = inc_with_drivers.groupby("driver_id").apply(
    lambda x: x[["incident_id","incident_type","severity","delivery_id"]].to_dict("records"),
    include_groups=False
).to_dict()

driver_docs = []
for _, row in drivers.iterrows():
    doc = row.to_dict()
    doc["incidents"] = inc_by_driver.get(doc["driver_id"], [])
    driver_docs.append(doc)

db.drivers.insert_many(driver_docs)
print(f"drivers: {db.drivers.count_documents({})} docs")

In [ ]:
# hubs - embed vehicles per hub using zone mapping
zone_to_hub = hubs.set_index("zone")["hub_id"].to_dict()
vehicles["assigned_hub_id"] = (vehicles["assigned_zone"]
    .str.strip().str.title().replace("Ctr","Central").map(zone_to_hub))

veh_by_hub = vehicles.groupby("assigned_hub_id").apply(
    lambda x: x[["vehicle_id","vehicle_type","battery_health_pct","telematics_version"]].to_dict("records"),
    include_groups=False
).to_dict()

hub_docs = []
for _, row in hubs.iterrows():
    doc = row.to_dict()
    doc["vehicles"] = veh_by_hub.get(doc["hub_id"], [])
    hub_docs.append(doc)

db.hubs.insert_many(hub_docs)
print(f"hubs: {db.hubs.count_documents({})} docs")

In [ ]:
# app_events - flat collection, nothing to embed
db.app_events.insert_many(app_events.to_dict("records"))
print(f"app_events: {db.app_events.count_documents({})} docs")

print("\nAll collections:")
for name in ["customer_profiles","deliveries","drivers","hubs","app_events"]:
    print(f"  {name}: {db[name].count_documents({})} docs")

In [ ]:
# READ - failed deliveries in North zone
for doc in list(db.deliveries.find(
    {"delivery_status": "Failed", "pickup_zone": "North"},
    {"delivery_id": 1, "driver_id": 1, "pickup_zone": 1, "_id": 0}
))[:5]:
    print(doc)

In [ ]:
# READ - customers with open complaints
for doc in list(db.customer_profiles.find(
    {"complaints.status": "Open"},
    {"customer_id": 1, "complaints": 1, "_id": 0}
))[:3]:
    print(doc)

In [ ]:
# READ - high risk drivers with 3 or more incidents
for doc in list(db.drivers.find(
    {"incidents.2": {"$exists": True}},
    {"driver_id": 1, "incidents": 1, "_id": 0}
))[:3]:
    print(doc)

In [ ]:
# UPDATE - set max capacity on hub H01 and flag priority customers
db.hubs.update_one({"hub_id": "H01"}, {"$set": {"max_capacity": 60}})
result = db.customer_profiles.update_many(
    {"complaints": {"$elemMatch": {"severity": "High", "status": "Open"}}},
    {"$set": {"priority_customer": True}}
)
print(f"flagged {result.modified_count} priority customers")

In [ ]:
# DELETE - remove app events with no zone context
result = db.app_events.delete_many({"zone_context": None})
print(f"deleted {result.deleted_count} incomplete app events")

In [ ]:
# AGGREGATE - failure rate by zone
for doc in db.deliveries.aggregate([
    {"$group": {
        "_id": "$pickup_zone",
        "total": {"$sum": 1},
        "failed": {"$sum": {"$cond": [{"$eq": ["$delivery_status","Failed"]}, 1, 0]}}
    }},
    {"$addFields": {"failure_rate": {"$multiply": [{"$divide": ["$failed","$total"]}, 100]}}},
    {"$sort": {"failure_rate": -1}}
]):
    print(doc)

In [ ]:
# AGGREGATE - top 10 drivers by total failures
for doc in db.deliveries.aggregate([
    {"$group": {
        "_id": "$driver_id",
        "avg_rating": {"$avg": "$customer_rating_post_delivery"},
        "total_failed": {"$sum": {"$cond": [{"$eq": ["$delivery_status","Failed"]}, 1, 0]}}
    }},
    {"$sort": {"total_failed": -1}},
    {"$limit": 10}
]):
    print(doc)

In [ ]:
# AGGREGATE - complaint breakdown by type and severity
for doc in db.customer_profiles.aggregate([
    {"$unwind": "$complaints"},
    {"$group": {
        "_id": {"type": "$complaints.complaint_type", "severity": "$complaints.severity"},
        "count": {"$sum": 1}
    }},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]):
    print(doc)

In [ ]:
# AGGREGATE - hub performance via $lookup
for doc in db.hubs.aggregate([
    {"$lookup": {"from":"deliveries","localField":"zone","foreignField":"pickup_zone","as":"zone_deliveries"}},
    {"$addFields": {
        "vehicle_count":     {"$size": "$vehicles"},
        "total_deliveries":  {"$size": "$zone_deliveries"},
        "failed_deliveries": {"$size": {"$filter": {
            "input": "$zone_deliveries", "as": "d",
            "cond": {"$eq": ["$$d.delivery_status","Failed"]}
        }}}
    }},
    {"$project": {"hub_id":1,"zone":1,"vehicle_count":1,"total_deliveries":1,"failed_deliveries":1,"_id":0}},
    {"$sort": {"failed_deliveries": -1}}
]):
    print(doc)

In [ ]:
# create indexes on the most queried fields
db.deliveries.create_index([("delivery_status", 1)])
db.deliveries.create_index([("pickup_zone", 1)])
db.deliveries.create_index([("driver_id", 1)])
db.deliveries.create_index([("pickup_zone", 1), ("delivery_status", 1)])
db.customer_profiles.create_index([("complaints.status", 1)])
print("indexes:", list(db.deliveries.index_information().keys()))

In [ ]:
# time query before and after indexing + explain plan
start = time.time()
list(db.deliveries.find({"delivery_status": "Failed", "pickup_zone": "North"}))
print(f"before: {(time.time()-start)*1000:.2f}ms")

start = time.time()
list(db.deliveries.find({"delivery_status": "Failed", "pickup_zone": "North"}))
print(f"after:  {(time.time()-start)*1000:.2f}ms")

explain = db.deliveries.find({"delivery_status": "Failed", "pickup_zone": "North"}).explain()
print("winning plan:", explain["queryPlanner"]["winningPlan"])